In [1]:
# ============================================================
# INITIALIZE RAG PIPELINE
# ============================================================

from pathlib import Path
import sys

# Notebook:
# rag/Test/Test.ipynb
CURRENT_DIR = Path.cwd()
RAG_DIR = CURRENT_DIR.parent

if str(RAG_DIR) not in sys.path:
    sys.path.insert(0, str(RAG_DIR))


# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------

from retrieval.hybrid_retriever import HybridRetriever
from retrieval.reranker import CrossEncoderReranker
from vector_store.Metadata_filter import MetadataFilter

from context.context_selector import ContextSelector
from context.context_builder import ContextBuilder


# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

CHUNKS_FILE = (
    RAG_DIR
    / "embeddings"
    / "processed"
    / "chunks"
    / "chunks.json"
)


# ------------------------------------------------------------
# INITIALIZE
# ------------------------------------------------------------

print("Initializing Hybrid Retriever...")
hybrid_retriever = HybridRetriever()

print("Initializing Reranker...")
reranker = CrossEncoderReranker()

print("Initializing Metadata Filter...")
metadata_filter = MetadataFilter(
    chunks_file=str(CHUNKS_FILE)
)

print("Initializing Context Selector...")
context_selector = ContextSelector(
    top_k=3
)

print("Initializing Context Builder...")
context_builder = ContextBuilder()


print("\n" + "=" * 70)
print("RAG COMPONENTS READY")
print("=" * 70)

C:\Users\Ahmed\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing Hybrid Retriever...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  29%|██▊       | 57/199 [00:00<00:00, 547.24it/s]

Loading weights:  56%|█████▋    | 112/199 [00:00<00:00, 478.44it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 774.78it/s]

Initializing Reranker...
CROSS-ENCODER RERANKER
Loading model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1132.49it/s]

Model loaded successfully.
Initializing Metadata Filter...
Loaded 722 chunks.
Initializing Context Selector...
Initializing Context Builder...

RAG COMPONENTS READY


In [2]:
from pprint import pprint

query = "How does oxygen concentration affect flame behavior in microgravity?"
results = hybrid_retriever.search(query, top_k=3)

print("Results type:", type(results))
print("Number of results:", len(results))

if results:
    first = results[0]

    print("\nFirst result type:", type(first))

    if isinstance(first, dict):
        print("Top-level keys:", list(first.keys()))

        for key, value in first.items():
            print(f"\n--- {key} ---")
            print("Type:", type(value))

            if isinstance(value, dict):
                pprint(value)
            elif isinstance(value, str):
                print(value[:1500])
            else:
                pprint(value)

    else:
        pprint(first)

Results type: <class 'list'>
Number of results: 3

First result type: <class 'dict'>
Top-level keys: ['chunk_id', 'rrf_score', 'dense_rank', 'bm25_rank', 'dense_score', 'bm25_score', 'hybrid_rank']

--- chunk_id ---
Type: <class 'int'>
720

--- rrf_score ---
Type: <class 'float'>
0.03200204813108039

--- dense_rank ---
Type: <class 'int'>
2

--- bm25_rank ---
Type: <class 'int'>
3

--- dense_score ---
Type: <class 'float'>
0.7630226016044617

--- bm25_score ---
Type: <class 'float'>
6.94641823500619

--- hybrid_rank ---
Type: <class 'int'>
1


In [3]:

# Inspect the objects used by your existing retrieval pipeline
print("Retriever type:", type(hybrid_retriever))

print("\nRetriever attributes:")
print([
    name for name in dir(hybrid_retriever)
    if not name.startswith("_")
])


Retriever type: <class 'retrieval.hybrid_retriever.HybridRetriever'>

Retriever attributes:
['bm25_retriever', 'dense_retriever', 'reciprocal_rank_fusion', 'rrf_k', 'search']


In [4]:

print("=== DENSE RETRIEVER ===")
print("Type:", type(hybrid_retriever.dense_retriever))
print("Attributes:", [
    name for name in dir(hybrid_retriever.dense_retriever)
    if not name.startswith("_")
])

print("\n=== BM25 RETRIEVER ===")
print("Type:", type(hybrid_retriever.bm25_retriever))
print("Attributes:", [
    name for name in dir(hybrid_retriever.bm25_retriever)
    if not name.startswith("_")
])


=== DENSE RETRIEVER ===
Type: <class 'retrieval.dense_retriever.DenseRetriever'>
Attributes: ['encode_query', 'index', 'index_file', 'model', 'search']

=== BM25 RETRIEVER ===
Type: <class 'retrieval.bm25_retriever.BM25Retriever'>
Attributes: ['bm25', 'chunks', 'chunks_file', 'search', 'tokenize', 'tokenized_corpus']


In [5]:

bm25_chunks = hybrid_retriever.bm25_retriever.chunks

def inspect_retrieval(query, top_k=2):
    results = hybrid_retriever.search(query, top_k=top_k)

    # Map the actual chunk_id to its original chunk
    chunks_by_id = {
        chunk["chunk_id"]: chunk
        for chunk in bm25_chunks
    }

    print(f"\nQUERY: {query}")
    print("=" * 90)

    for result in results:
        chunk_id = result["chunk_id"]
        chunk = chunks_by_id.get(chunk_id)

        if chunk is None:
            print(f"\nChunk ID {chunk_id}: no matching chunk found")
            continue

        metadata = chunk.get("metadata", {})

        print(f"\nHybrid rank: {result.get('hybrid_rank')}")
        print(f"Chunk ID: {chunk_id}")
        print(f"Experiment: {metadata.get('experiment_id')}")
        print(f"Chunk type: {metadata.get('chunk_type')}")
        print(f"Fuel: {metadata.get('fuel_mix')}")
        print(f"Oxygen (%): {metadata.get('oxygen_before_pct')}")
        print(f"Pressure (kPa): {metadata.get('pressure_kpa')}")
        print(f"Burn time (s): {metadata.get('burn_time_s')}")
        print(f"Outcome: {metadata.get('outcome')}")
        print(f"Source: {metadata.get('source')}")
        print("Text preview:", chunk.get("text", "")[:300].replace("\n", " "))


queries = [
    "How does oxygen concentration affect flame behavior in microgravity?",
    "What relationship exists between oxygen concentration and burn time?",
    "Compare tests that self-extinguished with tests still burning at fuel cut-off.",
    "Which experiments have the longest burn times?"
]

for query in queries:
    inspect_retrieval(query, top_k=2)




QUERY: How does oxygen concentration affect flame behavior in microgravity?

Hybrid rank: 1
Chunk ID: 720
Experiment: SUMMARY-OXYGEN
Chunk type: dataset_summary
Fuel: None
Oxygen (%): None
Pressure (kPa): None
Burn time (s): None
Outcome: None
Source: SUMMARY-OXYGEN.md
Text preview: # Effect of oxygen concentration — all tests  ## Summary  - **Air-like oxygen (18-23%)** (64 tests, O2 18.0-23.4%): 41 ignited, 41 self-extinguished, 0 still burning when the fuel flow was cut; median burn time 8.0 s, longest 20.0 s; mean hazard index 10.9/100. Longest-burning tests: CFI-21106-G6, C

Hybrid rank: 2
Chunk ID: 717
Experiment: SUMMARY-DATASET
Chunk type: dataset_summary
Fuel: None
Oxygen (%): None
Pressure (kPa): None
Burn time (s): None
Outcome: None
Source: SUMMARY-DATASET.md
Text preview: # Dataset overview — NASA PSI-159 CFI-G Normal Flames  ## Summary  The knowledge base contains 179 real tests from the NASA PSI-159 CFI-G (Cool Flames Investigation with Gases) experiment on the Internati


QUERY: What relationship exists between oxygen concentration and burn time?

Hybrid rank: 1
Chunk ID: 720
Experiment: SUMMARY-OXYGEN
Chunk type: dataset_summary
Fuel: None
Oxygen (%): None
Pressure (kPa): None
Burn time (s): None
Outcome: None
Source: SUMMARY-OXYGEN.md
Text preview: # Effect of oxygen concentration — all tests  ## Summary  - **Air-like oxygen (18-23%)** (64 tests, O2 18.0-23.4%): 41 ignited, 41 self-extinguished, 0 still burning when the fuel flow was cut; median burn time 8.0 s, longest 20.0 s; mean hazard index 10.9/100. Longest-burning tests: CFI-21106-G6, C

Hybrid rank: 2
Chunk ID: 565
Experiment: CFI-21162-A4
Chunk type: experimental_conditions
Fuel: 30% ethane
Oxygen (%): 37.21
Pressure (kPa): 101.3
Burn time (s): 304.0
Outcome: self_extinguished
Source: CFI-21162-A4.md
Text preview: # Experiment CFI-21162-A4 — 30% ethane, O2 37.21%, 101.3 kPa  ## Experimental Conditions  - **Oxygen Concentration (before test):** 37.21 % - **Oxygen Concentration (after test):**


QUERY: Which experiments have the longest burn times?

Hybrid rank: 1
Chunk ID: 719
Experiment: SUMMARY-FUELS
Chunk type: dataset_summary
Fuel: None
Oxygen (%): None
Pressure (kPa): None
Burn time (s): None
Outcome: None
Source: SUMMARY-FUELS.md
Text preview: # Comparison of fuels — ethane, propane, butane  ## Summary  By fuel: - **ethane** (41 tests, O2 18.0-40.0%): 40 ignited, 30 self-extinguished, 10 still burning when the fuel flow was cut; median burn time 49.0 s, longest 304.0 s; mean hazard index 37.5/100. Longest-burning tests: CFI-21162-A4, CFI-

Hybrid rank: 2
Chunk ID: 721
Experiment: SUMMARY-PRESSURE
Chunk type: dataset_summary
Fuel: None
Oxygen (%): None
Pressure (kPa): None
Burn time (s): None
Outcome: None
Source: SUMMARY-PRESSURE.md
Text preview: # Effect of pressure — all tests  ## Summary  - **51 kPa** (25 tests, O2 20.7-39.3%): 20 ignited, 16 self-extinguished, 4 still burning when the fuel flow was cut; median burn time 60.0 s, longest 274.0 s; mean hazard index 31